# E9 — State-aware non-graph baselines for EV battery disassembly

This notebook tests whether **local process-state summaries** can reproduce the
reference-ranking improvement attributed to DynamicReal. It was prepared from
your uploaded `graph-ev-battery-revised.7z`, specifically the E6 v2 and E3 notebooks.
It is self-contained: you do not need to clone or rerun the original repository.

| Condition | Learner | Structural information supplied |
|---|---|---|
| StaticFeatureMLP | Capacity-matched MLP | Initial local fixation summaries |
| StateAwareMLP | Same MLP, same width | Post-current local fixation summaries |
| StaticFeatureXGB | 250-tree XGBRanker | Initial local fixation summaries |
| StateAwareXGB | Same tree configuration | Post-current local fixation summaries |
| DynamicRealGNN | Original E6 GNN rerun | Full post-current typed fixation graph |

StaticFeature controls include the same shared-component and categorical inputs
as their state-aware counterparts. Their state-summary fields stay at the initial
values. This is a matched **state-information** comparison, rather than a rerun
of the old attribute-only MLP/XGB results.

**Run in Colab:** upload this `.ipynb` through **File → Upload notebook**.
Choose a GPU runtime (A100, L4, or T4 all work), then run the numbered sections
from top to bottom. The public KIT dataset is downloaded automatically if an
existing copy is not found. The original repository archive is not needed.

**Default full run: 150 fits.** Four non-graph conditions × 10 products ×
3 seeds, plus 30 DynamicReal fits. Static controls and the GNN rerun can be
switched off in Cell 2 for a shorter descriptive comparison. Keep the defaults
for the strongest manuscript experiment.

Results are written after **every product/seed/condition fit**. With Google
Drive enabled, reconnect after a Colab interruption, rerun the setup cells,
and run Cell 10 again: completed fits are reused. An interrupted fit restarts;
earlier finished fits remain intact.

**Interpretation:** these are teacher-forced reference next-operation rankings,
not feasibility or closed-loop planning results. State scalars come from the
initial product record and the executed prefix only. Raw identifiers, candidate
positions, `next_op_id`, and visibility annotations are never model inputs.
The current operation is applied before candidate features are computed.

**Preparation checks:** validated on the original ten-product dataset using CPU short fits (one neural epoch and five trees), including E6 state agreement, candidate reordering, checkpoint reuse, summaries, and export. The full manuscript experiment has not been run in this file.

## Cell 1 — Dependencies and imports
Install only missing dependencies; do not replace Colab's PyTorch installation
or restart the runtime. XGBoost 2.x/3.x and scikit-learn 1.2+ are supported.

In [ ]:
import sys, importlib.util, subprocess
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'torch': 'torch',
          'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.2',
          'xgboost': 'xgboost>=2,<4', 'matplotlib': 'matplotlib', 'packaging': 'packaging',
          'IPython': 'ipython'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import os, gc, io, json, math, random, time, tarfile, zipfile
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import hashlib, itertools, shutil, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.preprocessing import OneHotEncoder
from packaging.version import Version
import torch
import torch.nn as nn
import torch.nn.functional as F
import xgboost as xgb
import matplotlib.pyplot as plt
from IPython.display import display
if Version(xgb.__version__) < Version('2.0') or Version(sklearn.__version__) < Version('1.2'):
    raise RuntimeError('Install xgboost>=2 and scikit-learn>=1.2 in a clean Colab runtime.')
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
print({'python': sys.version.split()[0], 'torch': torch.__version__,
       'xgboost': xgb.__version__, 'sklearn': sklearn.__version__})

## Cell 2 — Fixed configuration and durable output directory
Leave the scientific settings unchanged for the full study. Drive authorization
is Colab's normal connection step. If you use an already-mounted Drive it is reused.
Setting `USE_DRIVE=False` saves locally and requires downloading the result ZIP
before a runtime reset. The notebook never deletes earlier results.

In [ ]:
SEEDS = [17, 29, 43]
EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32
BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119
DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE
XGB_TREES = 250
CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RUN_STATIC_CONTROLS = True
RUN_DYNAMICREAL = True
RUN_FULL_EXPERIMENT = True
SAVE_MODELS = True
USE_DRIVE = True
AUTO_DOWNLOAD_DATASET = True
DATA_PATH = ''  # Optional: directory with the four CSVs, or the KIT .tar archive.
LOCAL_OUTPUT_ROOT = '/content/ev_state_baselines' if IN_COLAB else './ev_state_baselines'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/EV_State_Aware_Baselines'

SOURCE_E6_SHA256 = '90c01dc01b2eccfb9e592de3ba649c7017a52086d7ee23260db992cb47d48d81'
IMPLEMENTATION_SHA256 = '881566fa41124575a6d5ed0f99cb08b518548b4746e32e87e34249164c4f4483'
FEATURE_VERSION = 'E9-state-scalars-v1.0'
METRICS = ['hits1', 'hits3', 'mrr']
ROLE_NAMES = ['source', 'target', 'part']

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive is not mounted. Retry Cell 2, or explicitly set USE_DRIVE=False.')
    OUTPUT_ROOT = Path(DRIVE_OUTPUT_ROOT)
else:
    OUTPUT_ROOT = Path(LOCAL_OUTPUT_ROOT).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONDITIONS = ['StateAwareMLP', 'StateAwareXGB']
if RUN_STATIC_CONTROLS:
    CONDITIONS = ['StaticFeatureMLP', 'StateAwareMLP', 'StaticFeatureXGB', 'StateAwareXGB']
if RUN_DYNAMICREAL:
    CONDITIONS += ['DynamicRealGNN']
print('Neural device:', DEVICE)
print('Tree device: CPU histogram; threads:', CPU_THREADS)
print('Output directory:', OUTPUT_ROOT)
print('Conditions:', CONDITIONS)

## Cell 3 — Locate or download the original KIT data
The data are downloaded from the archive's publicly advertised content URL:
[dataset DOI](https://doi.org/10.35097/emz24pksshndq468).
Existing CSVs or the original `.tar` under `/content` are reused first.
File hashes are recorded so outputs from different datasets cannot be mixed.

In [ ]:
CSV_NAMES = ['battery_catalogue.csv', 'parts.csv', 'fixations.csv', 'operations.csv']
DATASET_URL = 'https://radar.kit.edu/radar-backend/archives/emz24pksshndq468/versions/1/content'
search_root = Path('/content') if IN_COLAB else Path.cwd()
def search_local_csvs(root):
    for folder, directories, filenames in os.walk(root):
        directories[:] = [d for d in directories if d not in {'drive', '.git', 'node_modules', '__pycache__'}]
        if set(CSV_NAMES).issubset(filenames):
            return Path(folder)
    return None
def extract_dataset_archive(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:*') as tf:
        resolved = destination.resolve()
        for member in tf.getmembers():
            target = (resolved/member.name).resolve()
            if not target.is_relative_to(resolved) or member.issym() or member.islnk():
                raise ValueError('Unsafe archive entry: '+member.name)
        if sys.version_info >= (3, 12):
            tf.extractall(destination, filter='data')
        else:
            tf.extractall(destination)
    result = search_local_csvs(destination)
    if result is None:
        raise FileNotFoundError('Four dataset CSVs were not found in the archive.')
    return result
if DATA_PATH:
    supplied = Path(DATA_PATH).expanduser()
    if supplied.is_file():
        DATA = extract_dataset_archive(supplied, search_root/'ev_kit_dataset')
    elif supplied.is_dir():
        DATA = search_local_csvs(supplied)
    else:
        raise FileNotFoundError(DATA_PATH)
else:
    DATA = search_local_csvs(search_root)
if DATA is None:
    archives = list(search_root.glob('10.35097-*.tar'))
    if archives:
        DATA = extract_dataset_archive(archives[0], search_root/'ev_kit_dataset')
    elif AUTO_DOWNLOAD_DATASET:
        archive = search_root/'10.35097-emz24pksshndq468.tar'
        if not archive.exists():
            print('Downloading the public KIT dataset ...')
            temp = archive.with_suffix('.tar.part')
            request = urllib.request.Request(DATASET_URL, headers={'User-Agent': 'EV-research-notebook/1.0'})
            with urllib.request.urlopen(request, timeout=90) as response, temp.open('wb') as stream:
                shutil.copyfileobj(response, stream)
            os.replace(temp, archive)
        DATA = extract_dataset_archive(archive, search_root/'ev_kit_dataset')
    else:
        raise FileNotFoundError('Upload the KIT .tar or set DATA_PATH to its extracted CSV directory.')
if DATA is None:
    raise FileNotFoundError('The configured folder does not contain the four KIT CSVs.')
DATASET_HASHES = {name: hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in CSV_NAMES}
bat = pd.read_csv(DATA/'battery_catalogue.csv')
parts = pd.read_csv(DATA/'parts.csv')
fix = pd.read_csv(DATA/'fixations.csv')
ops = pd.read_csv(DATA/'operations.csv')
# Preserve the column convention used by E6; accept case variations from a CSV export.
rename = {c: {'frompartid': 'fromPartID', 'topartid': 'toPartID'}.get(c.lower(), c) for c in fix.columns}
fix = fix.rename(columns=rename)
actual_counts = (len(bat), len(parts), len(fix), len(ops))
assert actual_counts == (10, 563, 1896, 2281), f'Unexpected dataset version/counts: {actual_counts}'
for frame, required in [(bat, ['id', 'name']), (parts, ['id', 'class', 'battery_id']),
                        (fix, ['id', 'class', 'battery_id', 'fromPartID', 'toPartID']),
                        (ops, ['id', 'skill_id', 'fixation_id', 'part_id', 'next_op_id'])]:
    assert set(required).issubset(frame.columns), set(required)-set(frame.columns)
    assert frame.id.is_unique and frame.id.notna().all()
assert (ops.fixation_id.notna() ^ ops.part_id.notna()).all()
print('Data:', DATA)
print('Counts (products, parts, fixations, operations):', actual_counts)
display(bat[['id', 'name']])

## Cell 4 — Original E6 chain, vocabulary, and operation representation
These helpers are taken from E6 v2. Categorical vocabularies use the nine
training products only; unseen categories map to `<UNK>`. `next_op_id` is used
solely to reconstruct reference supervision. It is absent from predictive inputs.

In [ ]:
battery_names = bat.set_index('id')['name'].to_dict()
fix_battery = fix.set_index('id')['battery_id'].to_dict()
part_battery = parts.set_index('id')['battery_id'].to_dict()
part_class_raw = parts.set_index('id')['class'].to_dict()
fix_info_raw = fix.set_index('id')[['class', 'fromPartID', 'toPartID']].to_dict('index')

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])
ops['battery_id'] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index('id')
chains = {}
for b in sorted(ops['battery_id'].unique()):
    sub = ops[ops['battery_id'] == b]
    ids = set(sub['id'].astype(int))
    next_ids = set(sub['next_op_id'].dropna().astype(int))
    starts = list(ids - next_ids)
    terminals = sub[sub['next_op_id'].isna()]['id'].astype(int).tolist()
    assert len(starts) == 1
    assert len(terminals) == 1
    seq = []
    seen = set()
    cur = int(starts[0])
    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)
        nxt = ops_by_id.loc[cur, 'next_op_id']
        if pd.isna(nxt):
            break
        cur = int(nxt)
    assert len(seq) == len(ids), f'Broken chain for battery {b}'
    chains[int(b)] = seq
print({battery_names[b]: len(seq) for b, seq in chains.items()})

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)
    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi['fromPartID'])])
        dst_cls = str(part_class_raw[int(fi['toPartID'])])
        return f"{skill}|fix:{fi['class']}|{src_cls}->{dst_cls}"
    return f'{skill}|part:{part_class_raw[int(r.part_id)]}'
raw_signatures = {b: [raw_full_signature(op_id) for op_id in seq] for b, seq in chains.items()}

def make_vocab(values):
    vals = sorted({str(x) for x in values if pd.notna(x)})
    return {'<UNK>': 0, **{v: i + 1 for i, v in enumerate(vals)}}

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {'part': make_vocab(parts[parts['battery_id'].isin(train_batteries)]['class']), 'fix': make_vocab(fix[fix['battery_id'].isin(train_batteries)]['class']), 'skill': make_vocab(ops[ops['battery_id'].isin(train_batteries)]['skill_id'])}

def build_battery_pack(battery_id, vocab, graph_mode='real', shuffle_id=0):
    assert graph_mode in {'real', 'shuffle', 'noedge'}
    psub = parts[parts['battery_id'] == battery_id].copy()
    fsub = fix[fix['battery_id'] == battery_id].copy()
    part_id_to_local = {int(pid): i for i, pid in enumerate(psub['id'].astype(int))}
    n_nodes = len(psub)
    true_degree = np.zeros(n_nodes, dtype=np.float32)
    edge_src = []
    edge_dst = []
    edge_type = []
    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]
        true_degree[a] += 1
        true_degree[b] += 1
        edge_src += [a, b]
        edge_dst += [b, a]
        et = lookup(vocab['fix'], r['class'])
        edge_type += [et, et]
    true_degree_norm = true_degree / max(1.0, float(true_degree.max()))
    node_part_class = np.array([lookup(vocab['part'], cls) for cls in psub['class']], dtype=np.int64)
    true_edge_index = np.array([edge_src, edge_dst], dtype=np.int64)
    true_edge_type = np.array(edge_type, dtype=np.int64)
    if graph_mode == 'real':
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()
    elif graph_mode == 'shuffle':
        rng_graph = np.random.default_rng(GRAPH_SHUFFLE_BASE + 10007 * int(shuffle_id) + int(battery_id))
        permutation = rng_graph.permutation(n_nodes)
        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()
        shuffled_degree = np.bincount(graph_edge_index.reshape(-1), minlength=n_nodes).astype(np.float32)
        graph_degree_norm = shuffled_degree / max(1.0, float(shuffled_degree.max()))
    else:
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)
        graph_degree_norm = np.zeros(n_nodes, dtype=np.float32)
    seq = [int(x) for x in chains[battery_id]]
    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []
    for op_id in seq:
        r = ops_by_id.loc[op_id]
        skill_idx = lookup(vocab['skill'], r.skill_id)
        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]
            src_pid = int(fi['fromPartID'])
            dst_pid = int(fi['toPartID'])
            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]
            categorical.append([skill_idx, 0, lookup(vocab['fix'], fi['class']), lookup(vocab['part'], part_class_raw[src_pid]), lookup(vocab['part'], part_class_raw[dst_pid]), 0])
            graph_nodes.append([src_node, dst_node, -1])
            numeric.append([true_degree_norm[src_node], true_degree_norm[dst_node], 0.0])
        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]
            categorical.append([skill_idx, 1, 0, 0, 0, lookup(vocab['part'], part_class_raw[pid])])
            graph_nodes.append([-1, -1, node])
            numeric.append([0.0, 0.0, true_degree_norm[node]])
        signatures.append(raw_full_signature(op_id))
    return {'battery_id': int(battery_id), 'battery_name': battery_names[int(battery_id)], 'node_part_class': node_part_class, 'node_degree': graph_degree_norm.astype(np.float32), 'edge_index': graph_edge_index, 'edge_type': graph_edge_type, 'op_categorical': np.asarray(categorical, dtype=np.int64), 'op_graph_nodes': np.asarray(graph_nodes, dtype=np.int64), 'op_numeric': np.asarray(numeric, dtype=np.float32), 'full_signatures': signatures, 'n_ops': len(seq), 'graph_mode': graph_mode, 'shuffle_id': int(shuffle_id)}

In [ ]:
FOLD_IDS = sorted(chains)
assert len(FOLD_IDS) == 10 and sum(len(seq)-1 for seq in chains.values()) == 2271
print('Planned full fits:', len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS))

## Cell 5 — Original E6 neural architecture and ranking convention
The DynamicReal rerun uses the original two-layer typed message-passing model,
eight negative candidates per training state, 30 epochs, one optimizer update
per training battery per epoch, and the original expected tie-aware metrics.

In [ ]:
class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(self, n_skill, n_fix, n_part, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)
        op_input_dim = emb_dim + 4 + emb_dim + emb_dim + emb_dim + emb_dim + 3
        self.op_mlp = nn.Sequential(nn.Linear(op_input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.score_mlp = nn.Sequential(nn.Linear(hidden_dim * 4 + 1, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(pack['op_categorical'], dtype=torch.long, device=DEVICE)
        nums = torch.as_tensor(pack['op_numeric'], dtype=torch.float32, device=DEVICE)
        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]
        x = torch.cat([self.skill_emb(skill), self.kind_emb(kind), self.fix_emb(fix_class), self.part_emb(src_class), self.part_emb(dst_class), self.part_emb(part_class), nums], dim=-1)
        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([current_repr, candidate_repr, torch.abs(current_repr - candidate_repr), current_repr * candidate_repr, progress], dim=-1)
        return self.score_mlp(x).squeeze(-1)

class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(self, n_part, n_fix, part_emb_dim=EMBED_DIM, edge_emb_dim=8, graph_dim=GNN_DIM, dropout=DROPOUT):
        super().__init__()
        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)
        self.input_proj = nn.Linear(part_emb_dim + 1, graph_dim)
        self.message_layers = nn.ModuleList([nn.Linear(graph_dim + edge_emb_dim, graph_dim) for _ in range(2)])
        self.self_layers = nn.ModuleList([nn.Linear(graph_dim, graph_dim) for _ in range(2)])
        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(pack['node_part_class'], dtype=torch.long, device=DEVICE)
        degree = torch.as_tensor(pack['node_degree'], dtype=torch.float32, device=DEVICE).unsqueeze(-1)
        h = F.relu(self.input_proj(torch.cat([self.part_emb(node_class), degree], dim=-1)))
        edge_index = torch.as_tensor(pack['edge_index'], dtype=torch.long, device=DEVICE)
        edge_type = torch.as_tensor(pack['edge_type'], dtype=torch.long, device=DEVICE)
        src = edge_index[0]
        dst = edge_index[1]
        edge_embedding = self.edge_emb(edge_type)
        for message_layer, self_layer in zip(self.message_layers, self.self_layers):
            msg = message_layer(torch.cat([h[src], edge_embedding], dim=-1))
            agg = torch.zeros((h.shape[0], h.shape[1]), dtype=h.dtype, device=DEVICE)
            agg.index_add_(0, dst, msg)
            counts = torch.zeros((h.shape[0], 1), dtype=h.dtype, device=DEVICE)
            counts.index_add_(0, dst, torch.ones((dst.numel(), 1), dtype=h.dtype, device=DEVICE))
            agg = agg / counts.clamp_min(1.0)
            h = F.relu(self_layer(h) + agg)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

class GNNOperationRanker(BaseOperationRanker):

    def __init__(self, n_skill, n_fix, n_part, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, graph_dim=GNN_DIM, dropout=DROPOUT):
        super().__init__(n_skill=n_skill, n_fix=n_fix, n_part=n_part, emb_dim=emb_dim, hidden_dim=hidden_dim, dropout=dropout)
        self.graph_encoder = GraphEncoder(n_part=n_part, n_fix=n_fix, part_emb_dim=emb_dim, graph_dim=graph_dim, dropout=dropout)
        self.structural_projection = nn.Sequential(nn.Linear(graph_dim * 3, hidden_dim), nn.ReLU())
        self.combine = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU())

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)
        op_nodes = torch.as_tensor(pack['op_graph_nodes'], dtype=torch.long, device=DEVICE)
        n_ops = op_nodes.shape[0]
        a = torch.zeros((n_ops, node_h.shape[1]), device=DEVICE)
        b = torch.zeros_like(a)
        fixation_mask = op_nodes[:, 0] >= 0
        if fixation_mask.any():
            a[fixation_mask] = node_h[op_nodes[fixation_mask, 0]]
            b[fixation_mask] = node_h[op_nodes[fixation_mask, 1]]
        part_mask = op_nodes[:, 2] >= 0
        if part_mask.any():
            a[part_mask] = node_h[op_nodes[part_mask, 2]]
        structural = self.structural_projection(torch.cat([a, b, torch.abs(a - b)], dim=-1))
        return self.combine(torch.cat([base, structural], dim=-1))

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = ([], [], [], [])
    for t in range(n_ops - 2):
        pool = np.arange(t + 2, n_ops, dtype=np.int64)
        neg = rng.choice(pool, size=k, replace=len(pool) < k)
        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(t / max(1, n_ops - 1))
    return (np.asarray(current, dtype=np.int64), np.asarray(positive, dtype=np.int64), np.stack(negative), np.asarray(progress, dtype=np.float32))

In [ ]:
def expected_tie_metrics(scores, target_position=0, tol=1e-08):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """
    scores = np.asarray(scores)
    target_score = scores[target_position]
    greater = int(np.sum(scores > target_score + tol))
    equal = int(np.sum(np.abs(scores - target_score) <= tol))
    equal = max(equal, 1)
    hits1 = max(0, min(equal, 1 - greater)) / equal if greater < 1 else 0.0
    hits3 = max(0, min(equal, 3 - greater)) / equal if greater < 3 else 0.0
    ranks = np.arange(greater + 1, greater + equal + 1, dtype=float)
    mrr = float(np.mean(1.0 / ranks))
    return (hits1, hits3, mrr)

def count_trainable_parameters(model):
    return sum((p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def training_signature_set(train_batteries):
    s = set()
    for b in train_batteries:
        s.update(raw_signatures[b])
    return s

## Cell 6 — Original E6 state updater and DynamicReal training
All original fixation records, including parallel fixations, remain represented.
Executed fixation operations remove their own records. An executed part is
marked complete and deactivated only when E6 finds no remaining active incident
modeled fixation. **This notebook deliberately retains E6's handling of `fixed`
connections without a separate operation**; it does not introduce a new updater
into the baseline comparison. Some completed components can remain structurally
active under that simplified convention.

In [ ]:
from collections import defaultdict
DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE

def _battery_structure_records(battery_id, vocab):
    psub = parts[parts['battery_id'] == battery_id].copy()
    fsub = fix[fix['battery_id'] == battery_id].copy()
    part_ids = psub['id'].astype(int).tolist()
    part_id_to_local = {pid: i for i, pid in enumerate(part_ids)}
    records = []
    for _, r in fsub.iterrows():
        records.append({'fixation_id': int(r['id']), 'u': part_id_to_local[int(r.fromPartID)], 'v': part_id_to_local[int(r.toPartID)], 'edge_type': lookup(vocab['fix'], r['class'])})
    return (psub, part_id_to_local, records)

def _dynamic_permutation(battery_id, n_nodes, shuffle_id):
    rng = np.random.default_rng(DYNAMIC_SHUFFLE_BASE + 10007 * int(shuffle_id) + int(battery_id))
    return rng.permutation(n_nodes)

def _state_edges_and_degree(n_nodes, records, removed_fixations, active_nodes, initial_max_degree):
    edge_src = []
    edge_dst = []
    edge_type = []
    degree = np.zeros(n_nodes, dtype=np.float32)
    for rec in records:
        if rec['fixation_id'] in removed_fixations:
            continue
        u = int(rec['u'])
        v = int(rec['v'])
        if not active_nodes[u] or not active_nodes[v]:
            continue
        degree[u] += 1.0
        degree[v] += 1.0
        edge_src += [u, v]
        edge_dst += [v, u]
        edge_type += [int(rec['edge_type']), int(rec['edge_type'])]
    if len(edge_src):
        edge_index = np.asarray([edge_src, edge_dst], dtype=np.int64)
        edge_type = np.asarray(edge_type, dtype=np.int64)
    else:
        edge_index = np.empty((2, 0), dtype=np.int64)
        edge_type = np.empty((0,), dtype=np.int64)
    degree_norm = (degree / max(1.0, float(initial_max_degree))).astype(np.float32)
    return (edge_index, edge_type, degree_norm)

def _complete_state_edges_and_degree(active_nodes):
    active = np.flatnonzero(active_nodes).astype(np.int64)
    n_nodes = len(active_nodes)
    if len(active) <= 1:
        return (np.empty((2, 0), dtype=np.int64), np.empty((0,), dtype=np.int64), np.zeros(n_nodes, dtype=np.float32))
    src = np.repeat(active, len(active))
    dst = np.tile(active, len(active))
    keep = src != dst
    src = src[keep]
    dst = dst[keep]
    edge_index = np.asarray([src, dst], dtype=np.int64)
    edge_type = np.zeros(len(src), dtype=np.int64)
    degree_norm = np.zeros(n_nodes, dtype=np.float32)
    degree_norm[active] = (len(active) - 1) / max(1, n_nodes - 1)
    return (edge_index, edge_type, degree_norm)

def _has_remaining_incident_fixation(node, records, removed_fixations, active_nodes):
    """
    Causal check: does this currently active node still participate in any
    modeled fixation that has not yet been executed and whose other endpoint
    is also structurally active?
    """
    for rec in records:
        if rec['fixation_id'] in removed_fixations:
            continue
        u = int(rec['u'])
        v = int(rec['v'])
        if not active_nodes[u] or not active_nodes[v]:
            continue
        if node == u or node == v:
            return True
    return False

def _deactivate_completed_parts(records, removed_fixations, active_nodes, part_completed):
    """
    Deactivate a part only after:
      1) its part operation has already occurred, and
      2) it has no remaining active incident fixation.

    Iterate to a fixed point because deactivating one completed isolated node
    can make another completed node isolated as well.
    """
    changed = True
    while changed:
        changed = False
        pending = np.flatnonzero(part_completed & active_nodes)
        for node in pending:
            node = int(node)
            if not _has_remaining_incident_fixation(node=node, records=records, removed_fixations=removed_fixations, active_nodes=active_nodes):
                active_nodes[node] = False
                changed = True
    return active_nodes

def build_e6_state_pack(battery_id, vocab, condition, shuffle_id=0):
    """
    Conditions:
      static_real
      dynamic_real
      dynamic_noedge
      dynamic_complete
      dynamic_shuffle

    Returns the invariant operation pack plus one block-diagonal graph
    containing all post-current ranking states t=0,...,n_ops-2.
    """
    assert condition in {'static_real', 'dynamic_real', 'dynamic_noedge', 'dynamic_complete', 'dynamic_shuffle'}
    base_pack = build_battery_pack(battery_id, vocab, graph_mode='real', shuffle_id=0)
    psub, part_id_to_local, true_records = _battery_structure_records(battery_id, vocab)
    n_nodes = len(psub)
    n_states = base_pack['n_ops'] - 1
    initial_degree = np.zeros(n_nodes, dtype=np.float32)
    for rec in true_records:
        initial_degree[rec['u']] += 1.0
        initial_degree[rec['v']] += 1.0
    initial_max_degree = max(1.0, float(initial_degree.max()))
    if condition == 'dynamic_shuffle':
        permutation = _dynamic_permutation(battery_id, n_nodes, shuffle_id)
        records = [{'fixation_id': rec['fixation_id'], 'u': int(permutation[rec['u']]), 'v': int(permutation[rec['v']]), 'edge_type': rec['edge_type']} for rec in true_records]
        event_node_map = permutation
    else:
        permutation = np.arange(n_nodes, dtype=np.int64)
        records = [dict(rec) for rec in true_records]
        event_node_map = permutation
    static_edge_index, static_edge_type, static_degree = _state_edges_and_degree(n_nodes=n_nodes, records=records, removed_fixations=set(), active_nodes=np.ones(n_nodes, dtype=bool), initial_max_degree=initial_max_degree)
    removed_fixations = set()
    active_nodes = np.ones(n_nodes, dtype=bool)
    part_completed = np.zeros(n_nodes, dtype=bool)
    state_edge_indices = []
    state_edge_types = []
    state_degrees = []
    state_active_fraction = []
    state_edge_fraction = []
    initial_undirected_edges = len(true_records)
    seq = [int(x) for x in chains[battery_id]]
    for t in range(n_states):
        op_id = seq[t]
        r = ops_by_id.loc[op_id]
        if condition != 'static_real':
            if pd.notna(r.fixation_id):
                removed_fixations.add(int(r.fixation_id))
            else:
                semantic_node = part_id_to_local[int(r.part_id)]
                structural_node = int(event_node_map[semantic_node])
                part_completed[structural_node] = True
            active_nodes = _deactivate_completed_parts(records=records, removed_fixations=removed_fixations, active_nodes=active_nodes, part_completed=part_completed)
        if condition == 'static_real':
            edge_index = static_edge_index
            edge_type = static_edge_type
            degree_norm = static_degree
            active_fraction = 1.0
            edge_fraction = 1.0
        elif condition == 'dynamic_noedge':
            edge_index = np.empty((2, 0), dtype=np.int64)
            edge_type = np.empty((0,), dtype=np.int64)
            degree_norm = np.zeros(n_nodes, dtype=np.float32)
            active_fraction = float(active_nodes.mean())
            edge_fraction = 0.0
        elif condition == 'dynamic_complete':
            edge_index, edge_type, degree_norm = _complete_state_edges_and_degree(active_nodes)
            active_fraction = float(active_nodes.mean())
            max_directed = n_nodes * max(0, n_nodes - 1)
            edge_fraction = edge_index.shape[1] / max(1, max_directed)
        else:
            edge_index, edge_type, degree_norm = _state_edges_and_degree(n_nodes=n_nodes, records=records, removed_fixations=removed_fixations, active_nodes=active_nodes, initial_max_degree=initial_max_degree)
            active_fraction = float(active_nodes.mean())
            edge_fraction = edge_index.shape[1] / 2.0 / max(1, initial_undirected_edges)
        state_edge_indices.append(edge_index)
        state_edge_types.append(edge_type)
        state_degrees.append(degree_norm)
        state_active_fraction.append(active_fraction)
        state_edge_fraction.append(edge_fraction)
    block_src = []
    block_dst = []
    block_type = []
    for state_idx, (edge_index, edge_type) in enumerate(zip(state_edge_indices, state_edge_types)):
        offset = state_idx * n_nodes
        if edge_index.shape[1]:
            block_src.append(edge_index[0] + offset)
            block_dst.append(edge_index[1] + offset)
            block_type.append(edge_type)
    if block_src:
        block_edge_index = np.asarray([np.concatenate(block_src), np.concatenate(block_dst)], dtype=np.int64)
        block_edge_type = np.concatenate(block_type).astype(np.int64)
    else:
        block_edge_index = np.empty((2, 0), dtype=np.int64)
        block_edge_type = np.empty((0,), dtype=np.int64)
    block_pack = {'node_part_class': np.tile(base_pack['node_part_class'], n_states).astype(np.int64), 'node_degree': np.concatenate(state_degrees).astype(np.float32), 'edge_index': block_edge_index, 'edge_type': block_edge_type}
    return {'base_pack': base_pack, 'state_graph_pack': block_pack, 'n_nodes': int(n_nodes), 'n_states': int(n_states), 'state_active_fraction': np.asarray(state_active_fraction, dtype=np.float32), 'state_edge_fraction': np.asarray(state_edge_fraction, dtype=np.float32), 'condition': condition, 'shuffle_id': int(shuffle_id), 'permutation': permutation.astype(np.int64)}

def build_e6_model(vocab):
    return GNNOperationRanker(n_skill=len(vocab['skill']), n_fix=len(vocab['fix']), n_part=len(vocab['part'])).to(DEVICE)

def compose_state_operation_repr(model, base_repr, node_h, base_pack, n_nodes, state_indices, op_indices):
    state_indices = torch.as_tensor(state_indices, dtype=torch.long, device=DEVICE)
    op_indices = torch.as_tensor(op_indices, dtype=torch.long, device=DEVICE)
    op_nodes_all = torch.as_tensor(base_pack['op_graph_nodes'], dtype=torch.long, device=DEVICE)
    op_nodes = op_nodes_all[op_indices]
    offsets = state_indices * int(n_nodes)
    n_items = len(op_indices)
    a = torch.zeros((n_items, node_h.shape[1]), dtype=node_h.dtype, device=DEVICE)
    b = torch.zeros_like(a)
    fixation_mask = op_nodes[:, 0] >= 0
    if fixation_mask.any():
        a[fixation_mask] = node_h[offsets[fixation_mask] + op_nodes[fixation_mask, 0]]
        b[fixation_mask] = node_h[offsets[fixation_mask] + op_nodes[fixation_mask, 1]]
    part_mask = op_nodes[:, 2] >= 0
    if part_mask.any():
        a[part_mask] = node_h[offsets[part_mask] + op_nodes[part_mask, 2]]
    structural = model.structural_projection(torch.cat([a, b, torch.abs(a - b)], dim=-1))
    return model.combine(torch.cat([base_repr[op_indices], structural], dim=-1))

In [ ]:
def train_one_e6_condition(train_batteries, test_battery, seed, condition, shuffle_id=0):
    set_seed(seed)
    vocab = build_fold_vocab(train_batteries)
    packs = {b: build_e6_state_pack(b, vocab, condition=condition, shuffle_id=shuffle_id) for b in train_batteries + [test_battery]}
    model = build_e6_model(vocab)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)
        model.train()
        for b in order:
            item = packs[b]
            base_pack = item['base_pack']
            n_ops = base_pack['n_ops']
            cur, pos, neg, prog = make_training_pairs(n_ops, NEGATIVES_PER_STATE, rng)
            state_idx = cur.copy()
            base_repr = model.base_operation_encoding(base_pack)
            node_h = model.graph_encoder(item['state_graph_pack'])
            cur_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=state_idx, op_indices=cur)
            pos_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=state_idx, op_indices=pos)
            prog_t = torch.as_tensor(prog, dtype=torch.float32, device=DEVICE).unsqueeze(-1)
            pos_score = model.score_pairs(cur_repr, pos_repr, prog_t)
            k = neg.shape[1]
            neg_flat = neg.reshape(-1)
            neg_state = np.repeat(state_idx, k)
            neg_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=neg_state, op_indices=neg_flat)
            cur_rep = cur_repr[:, None, :].expand(-1, k, -1).reshape(-1, cur_repr.shape[1])
            prog_rep = prog_t[:, None, :].expand(-1, k, -1).reshape(-1, 1)
            neg_score = model.score_pairs(cur_rep, neg_repr, prog_rep).reshape(len(cur), k)
            loss = F.softplus(-(pos_score[:, None] - neg_score)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return (model, packs[test_battery], vocab)

def evaluate_e6_model(model, item, train_batteries):
    model.eval()
    base_pack = item['base_pack']
    seen_signatures = training_signature_set(train_batteries)
    rows = []
    with torch.no_grad():
        base_repr = model.base_operation_encoding(base_pack)
        node_h = model.graph_encoder(item['state_graph_pack'])
        n_ops = base_pack['n_ops']
        for t in range(n_ops - 1):
            candidate_indices = np.arange(t + 1, n_ops, dtype=np.int64)
            state_current = np.asarray([t], dtype=np.int64)
            current_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=state_current, op_indices=np.asarray([t], dtype=np.int64))
            cand_states = np.full(len(candidate_indices), t, dtype=np.int64)
            candidate_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=cand_states, op_indices=candidate_indices)
            current_rep = current_repr.expand(len(candidate_indices), -1)
            progress = torch.full((len(candidate_indices), 1), t / max(1, n_ops - 1), dtype=torch.float32, device=DEVICE)
            scores = model.score_pairs(current_rep, candidate_repr, progress).cpu().numpy()
            h1, h3, mrr = expected_tie_metrics(scores, target_position=0)
            target_signature = base_pack['full_signatures'][t + 1]
            rows.append({'position': int(t), 'hits1': float(h1), 'hits3': float(h3), 'mrr': float(mrr), 'signature_seen_in_training': bool(target_signature in seen_signatures), 'active_node_fraction': float(item['state_active_fraction'][t]), 'edge_fraction': float(item['state_edge_fraction'][t])})
    detail = pd.DataFrame(rows)
    metrics = {'hits1': float(detail['hits1'].mean()), 'hits3': float(detail['hits3'].mean()), 'mrr': float(detail['mrr'].mean()), 'seen_signature_rate': float(detail['signature_seen_in_training'].mean())}
    return (metrics, detail)

## Cell 7 — Local state summaries and capacity-matched MLP

For each operation's source, target, or removed-part role, provide:
remaining fixation degree divided by the product's **initial** maximum degree;
fraction of that component's initial degree remaining; structural activity;
removal-complete status; and incident fixation-type histograms normalized by the
same initial maximum degree. Missing endpoint roles are zero.

Pair features add remaining-fixation, active-node, and completed-node fractions;
any shared component; shared-component count divided by two; and shared-component
Jaccard similarity. Initial operation degrees and categorical descriptors remain
available. Typed histograms use training-only vocabularies.

The baselines consume fixed-length summaries, **no adjacency matrix, node indices,
graph embeddings, neighbors' attributes, or message passing**. They deliberately
receive useful local state information so the comparison is strong. Width matching
uses only parameter counts, never held-out scores; static and state-aware MLPs
have identical architecture and capacity within a fold.

In [ ]:
def build_scalar_state_pack(battery_id, vocab):
    """Reproduce E6's post-current state without exposing adjacency to a learner."""
    base = build_battery_pack(battery_id, vocab, graph_mode="real")
    _, part_map, records = _battery_structure_records(battery_id, vocab)
    n = len(part_map)
    n_states = base["n_ops"] - 1
    n_types = len(vocab["fix"])
    roles = base["op_graph_nodes"]
    valid = roles >= 0
    safe_roles = np.maximum(roles, 0)
    initial = np.zeros(n, dtype=np.float32)
    for r in records:
        initial[r["u"]] += 1
        initial[r["v"]] += 1
    denom = max(1.0, float(initial.max()))
    removed = set()
    active = np.ones(n, dtype=bool)
    complete = np.zeros(n, dtype=bool)
    local_names = []
    for group in ["degree_norm", "degree_fraction", "active", "completed"]:
        local_names.extend(f"{group}_{role}" for role in ROLE_NAMES)
    type_labels = [k for k, _ in sorted(vocab["fix"].items(), key=lambda kv: kv[1])]
    for role in ROLE_NAMES:
        local_names.extend(f"incident_type_{role}_{label}" for label in type_labels)

    def snapshot():
        degree = np.zeros(n, dtype=np.float32)
        typed = np.zeros((n, n_types), dtype=np.float32)
        retained = 0
        for r in records:
            if r["fixation_id"] in removed or not (active[r["u"]] and active[r["v"]]):
                continue
            retained += 1
            for node in [r["u"], r["v"]]:
                degree[node] += 1
                typed[node, r["edge_type"]] += 1
        group = [degree / denom, degree / np.maximum(initial, 1.0),
                 active.astype(np.float32), complete.astype(np.float32)]
        local = [x[safe_roles] * valid for x in group]
        incident = typed[safe_roles] * valid[..., None] / denom
        local.append(incident.reshape(base["n_ops"], -1))
        local = np.concatenate(local, axis=1).astype(np.float32)
        global_features = np.array([retained / max(1, len(records)),
                                    active.mean(), complete.mean()], dtype=np.float32)
        return local, global_features, degree / denom

    initial_local, initial_global, _ = snapshot()
    local_states, global_states, node_degrees = [], [], []
    for t in range(n_states):
        row = ops_by_id.loc[int(chains[battery_id][t])]
        if pd.notna(row.fixation_id):
            removed.add(int(row.fixation_id))
        else:
            complete[part_map[int(row.part_id)]] = True
        _deactivate_completed_parts(records, removed, active, complete)
        local, global_features, degree = snapshot()
        local_states.append(local)
        global_states.append(global_features)
        node_degrees.append(degree)

    # Raw indices are used here only to calculate identity-overlap summaries.
    # They never become a model feature or a vocabulary category.
    overlaps = np.zeros((base["n_ops"], base["n_ops"], 3), dtype=np.float32)
    touched = [set(int(x) for x in row if x >= 0) for row in roles]
    for i, a in enumerate(touched):
        for j, b in enumerate(touched):
            shared = len(a & b)
            overlaps[i, j] = [float(shared > 0), shared / 2.0,
                              shared / max(1, len(a | b))]
    return {"base_pack": base, "local": np.stack(local_states),
            "global": np.stack(global_states), "initial_local": initial_local,
            "initial_global": initial_global, "overlaps": overlaps,
            "node_degrees": np.stack(node_degrees), "local_names": local_names,
            "n_nodes": n}

def scalar_inputs(item, states, operations, state_aware):
    states = np.asarray(states, dtype=np.int64)
    operations = np.asarray(operations, dtype=np.int64)
    if state_aware:
        return item["local"][states, operations]
    return item["initial_local"][operations]

def pair_extra(item, states, current, candidate, state_aware):
    states = np.asarray(states, dtype=np.int64)
    if state_aware:
        global_values = item["global"][states]
    else:
        global_values = np.repeat(item["initial_global"][None], len(states), axis=0)
    return np.concatenate([global_values, item["overlaps"][current, candidate]], axis=1)

class ScalarOperationRanker(BaseOperationRanker):
    """Shared operation MLP plus local state scalars; no graph/message passing."""
    def __init__(self, vocab, n_local, width):
        super().__init__(len(vocab["skill"]), len(vocab["fix"]), len(vocab["part"]),
                         hidden_dim=width)
        self.width = width
        self.op_mlp = nn.Sequential(nn.Linear(67 + n_local, width), nn.ReLU(),
                                    nn.Dropout(DROPOUT), nn.Linear(width, width), nn.ReLU())
        self.score_mlp = nn.Sequential(nn.Linear(width * 4 + 1 + 6, width), nn.ReLU(),
                                       nn.Dropout(DROPOUT), nn.Linear(width, 1))

    def encode_selected(self, item, states, operations, state_aware):
        base = item["base_pack"]
        operations = np.asarray(operations, dtype=np.int64)
        cats = torch.as_tensor(base["op_categorical"][operations], dtype=torch.long, device=DEVICE)
        nums = np.concatenate([base["op_numeric"][operations],
                               scalar_inputs(item, states, operations, state_aware)], axis=1)
        nums = torch.as_tensor(nums, dtype=torch.float32, device=DEVICE)
        values = [self.skill_emb(cats[:, 0]), self.kind_emb(cats[:, 1]),
                  self.fix_emb(cats[:, 2]), self.part_emb(cats[:, 3]),
                  self.part_emb(cats[:, 4]), self.part_emb(cats[:, 5]), nums]
        return self.op_mlp(torch.cat(values, dim=-1))

    def score_selected(self, item, states, current, candidate, state_aware,
                       current_repr=None, candidate_repr=None):
        cur = current_repr if current_repr is not None else self.encode_selected(item, states, current, state_aware)
        cand = candidate_repr if candidate_repr is not None else self.encode_selected(item, states, candidate, state_aware)
        progress = np.asarray(states, dtype=np.float32) / max(1, item["base_pack"]["n_ops"] - 1)
        extra = np.column_stack([progress, pair_extra(item, states, current, candidate, state_aware)])
        extra = torch.as_tensor(extra, dtype=torch.float32, device=DEVICE)
        return self.score_mlp(torch.cat([cur, cand, torch.abs(cur-cand), cur*cand, extra], dim=-1)).squeeze(-1)

def matched_mlp_width(vocab, n_local):
    """Analytical parameter matching; does not inspect any validation/test score."""
    target = count_trainable_parameters(GNNOperationRanker(
        len(vocab["skill"]), len(vocab["fix"]), len(vocab["part"])))
    # Three embedding tables plus the 2-by-4 kind table.
    embeddings = 12 * (len(vocab["skill"]) + len(vocab["fix"]) + len(vocab["part"])) + 8
    counts = {w: embeddings + 5*w*w + (67+n_local+11)*w + 1 for w in range(32, 161)}
    width = min(counts, key=lambda w: (abs(counts[w] - target), w))
    actual = count_trainable_parameters(ScalarOperationRanker(vocab, n_local, width))
    assert actual == counts[width], (actual, counts[width])
    return width, actual, target

## Cell 8 — Matched non-graph training and order-invariant evaluation
The MLP uses E6's optimizer, seeds, negative sampling, and product-weighted
update schedule. XGB uses E3's fixed 250-tree configuration and all nonsingleton
training candidates; tree and neural training budgets are distinct and recorded.
Candidates are constructed as inventory minus the executed prefix, then randomly
reordered. The target index is used only by the loss/metric, never as a feature.
Scoring uses a stable internal batch order, then scores are mapped to the shuffled
candidate list for metrics. This prevents tiny float32 batch-order differences
from changing near-ties at E6's original `1e-8` tie tolerance. Lookup indices
never enter the learner. Deterministic PyTorch operations are enabled and TF32
is disabled. Historical E6 remains descriptive only.

In [ ]:
def train_scalar_mlp(items, train_batteries, vocab, seed, state_aware, epochs=EPOCHS):
    set_seed(seed)
    n_local = items[train_batteries[0]]["local"].shape[-1]
    width, params, target = matched_mlp_width(vocab, n_local)
    # Width search must not consume the model initialization RNG.
    set_seed(seed)
    model = ScalarOperationRanker(vocab, n_local, width).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    losses = []
    for epoch in range(epochs):
        order = list(train_batteries)
        rng.shuffle(order)
        model.train()
        epoch_losses = []
        for b in order:
            item = items[b]
            cur, pos, neg, _ = make_training_pairs(item["base_pack"]["n_ops"], NEGATIVES_PER_STATE, rng)
            k = neg.shape[1]
            current_repr = model.encode_selected(item, cur, cur, state_aware)
            pos_score = model.score_selected(item, cur, cur, pos, state_aware, current_repr=current_repr)
            repeated_current = current_repr[:, None].expand(-1, k, -1).reshape(-1, model.width)
            neg_score = model.score_selected(item, np.repeat(cur, k), np.repeat(cur, k),
                                              neg.reshape(-1), state_aware,
                                              current_repr=repeated_current).reshape(len(cur), k)
            loss = F.softplus(-(pos_score[:, None] - neg_score)).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss for battery {b}, epoch {epoch+1}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        losses.append({"epoch": epoch+1, "loss": float(np.mean(epoch_losses))})
    return model, {"width": width, "parameter_count": params, "gnn_parameter_count": target,
                   "capacity_gap_fraction": (params-target)/target, "loss_curve": losses}

def make_scalar_encoder(vocab):
    op = [np.arange(len(vocab["skill"])), np.arange(2), np.arange(len(vocab["fix"]))]
    op += [np.arange(len(vocab["part"])) for _ in range(3)]
    encoder = OneHotEncoder(categories=op+op, handle_unknown="ignore",
                            sparse_output=True, dtype=np.float32)
    encoder.fit(np.zeros((1, 12), dtype=np.int64))
    return encoder

def scalar_pair_matrix(item, t, candidates, encoder, state_aware):
    candidates = np.asarray(candidates, dtype=np.int64)
    base = item["base_pack"]
    n = len(candidates)
    cur = np.full(n, t, dtype=np.int64)
    cur_cat = base["op_categorical"][cur]
    cand_cat = base["op_categorical"][candidates]
    cat = encoder.transform(np.column_stack([cur_cat, cand_cat]))
    cur_local = np.column_stack([base["op_numeric"][cur], scalar_inputs(item, cur, cur, state_aware)])
    cand_local = np.column_stack([base["op_numeric"][candidates], scalar_inputs(item, cur, candidates, state_aware)])
    numeric = np.column_stack([cur_local, cand_local, np.abs(cur_local-cand_local),
                               (cur_cat == cand_cat).astype(np.float32),
                               np.full(n, t/max(1, base["n_ops"]-1), dtype=np.float32),
                               pair_extra(item, cur, cur, candidates, state_aware)])
    return sparse.hstack([cat, sparse.csr_matrix(numeric)], format="csr", dtype=np.float32)

def training_xgb_matrix(items, train_batteries, encoder, state_aware):
    matrices, labels, queries = [], [], []
    rng = np.random.default_rng(BASE_SEED)
    qid = 0
    for b in train_batteries:
        item = items[b]
        n_ops = item["base_pack"]["n_ops"]
        for t in range(n_ops-2):
            candidates = rng.permutation(np.arange(t+1, n_ops))
            matrices.append(scalar_pair_matrix(item, t, candidates, encoder, state_aware))
            labels.append((candidates == t+1).astype(np.float32))
            queries.append(np.full(len(candidates), qid, dtype=np.int32))
            qid += 1
    return sparse.vstack(matrices, format="csr"), np.concatenate(labels), np.concatenate(queries)

def make_scalar_xgb(seed, n_estimators=XGB_TREES):
    return xgb.XGBRanker(objective="rank:pairwise", eval_metric="ndcg@3",
                         n_estimators=n_estimators, max_depth=6, learning_rate=0.05,
                         min_child_weight=1.0, subsample=0.90, colsample_bytree=0.90,
                         reg_lambda=1.0, reg_alpha=0.0, tree_method="hist",
                         n_jobs=CPU_THREADS, device="cpu", random_state=int(seed))

def candidate_inventory(n_ops, prefix):
    # Sequence indices are internal lookup keys, never numeric model inputs.
    return np.setdiff1d(np.arange(n_ops), np.asarray(prefix), assume_unique=True)

def evaluate_scalar_or_gnn(model, item, kind, state_aware=None, encoder=None, candidate_seed=None):
    base = item["base_pack"]
    n_ops = base["n_ops"]
    rng = np.random.default_rng(BASE_SEED + 1009 * int(base["battery_id"])
                                if candidate_seed is None else candidate_seed)
    if kind in {"mlp", "gnn"}:
        model.eval()
    rows = []
    with torch.no_grad():
        if kind == "gnn":
            base_repr = model.base_operation_encoding(base)
            node_h = model.graph_encoder(item["state_graph_pack"])
        for t in range(n_ops-1):
            inventory = candidate_inventory(n_ops, range(t+1))
            candidates = rng.permutation(inventory)
            # Fixed internal batching preserves E6's floating-point tie behavior.
            # Lookup order is never a feature. Shuffle scores back before the metric.
            scoring_candidates = inventory
            states = np.full(len(candidates), t, dtype=np.int64)
            if kind == "mlp":
                scores = model.score_selected(item, states, states, scoring_candidates, state_aware).cpu().numpy()
            elif kind == "xgb":
                scores = model.predict(scalar_pair_matrix(item, t, scoring_candidates, encoder, state_aware))
            elif kind == "gnn":
                current = compose_state_operation_repr(model, base_repr, node_h, base,
                            item["n_nodes"], np.array([t]), np.array([t])).expand(len(candidates), -1)
                candidate = compose_state_operation_repr(model, base_repr, node_h, base,
                            item["n_nodes"], states, scoring_candidates)
                progress = torch.full((len(candidates), 1), t/max(1, n_ops-1), device=DEVICE)
                scores = model.score_pairs(current, candidate, progress).cpu().numpy()
            else:
                raise ValueError(kind)
            if not np.isfinite(scores).all():
                raise FloatingPointError("Non-finite evaluation scores")
            scores = scores[np.searchsorted(inventory, candidates)]
            target = int(np.flatnonzero(candidates == t+1)[0])
            h1, h3, mrr = expected_tie_metrics(scores, target_position=target)
            rows.append({"position": t, "candidate_size": len(candidates),
                         "progress": t/max(1, n_ops-1), "hits1": h1, "hits3": h3, "mrr": mrr})
    detail = pd.DataFrame(rows)
    return {m: float(detail[m].mean()) for m in METRICS}, detail

In [ ]:
def validate_scalar_states(items, vocab):
    rows = []
    for b, item in items.items():
        real = build_e6_state_pack(b, vocab, "dynamic_real")
        degree = real["state_graph_pack"]["node_degree"].reshape(real["n_states"], real["n_nodes"])
        np.testing.assert_array_equal(degree, item["node_degrees"])
        np.testing.assert_allclose(real["state_edge_fraction"], item["global"][:, 0], atol=1e-7)
        np.testing.assert_allclose(real["state_active_fraction"], item["global"][:, 1], atol=1e-7)
        assert np.isfinite(item["local"]).all()
        assert item["local"].min() >= 0 and item["local"].max() <= 1.00001
        assert np.all(np.diff(item["global"][:, 0]) <= 1e-7)
        assert np.all(np.diff(item["global"][:, 1]) <= 1e-7)
        assert np.all(np.diff(item["global"][:, 2]) >= -1e-7)
        rows.append({"battery_id": b, "states": real["n_states"],
                     "local_features": item["local"].shape[-1], "e6_state_match": True,
                     "final_active_fraction": float(item["global"][-1, 1])})
        del real
    return pd.DataFrame(rows)

def assert_candidate_and_metric_invariants(item, vocab):
    encoder = make_scalar_encoder(vocab)
    for t in sorted(set([0, max(0, item["base_pack"]["n_ops"]//2-1), item["base_pack"]["n_ops"]-2])):
        candidates = candidate_inventory(item["base_pack"]["n_ops"], range(t+1))
        order = np.random.default_rng(89+t).permutation(len(candidates))
        for mode in [False, True]:
            x = scalar_pair_matrix(item, t, candidates, encoder, mode)
            xp = scalar_pair_matrix(item, t, candidates[order], encoder, mode)
            assert (xp-x[order]).nnz == 0
        scores = np.linspace(-1, 1, len(candidates))
        target = int(np.flatnonzero(candidates == t+1)[0])
        expected = expected_tie_metrics(scores, target)
        actual = expected_tie_metrics(scores[order], int(np.flatnonzero(order == target)[0]))
        np.testing.assert_allclose(expected, actual)
    np.testing.assert_allclose(expected_tie_metrics(np.ones(4), 2), [0.25, 0.75, np.mean(1/np.arange(1, 5))])
    np.testing.assert_allclose(expected_tie_metrics(np.ones(1)), [1, 1, 1])

## Cell 9 — Required preflight (does not write experiment results)
Check all ten products against E6's dynamic degrees, edge fractions, and activity
fractions. Check candidate-order invariance and tie metrics, then fit both MLPs,
both tree models, and optionally DynamicReal with reduced smoke-test budgets.
Smoke metrics are diagnostic only and are never added to the full result tables.

In [ ]:
probe_battery = FOLD_IDS[0]
probe_train = [b for b in FOLD_IDS if b != probe_battery]
probe_vocab = build_fold_vocab(probe_train)
probe_items = {b: build_scalar_state_pack(b, probe_vocab) for b in FOLD_IDS}
PREFLIGHT_DIAGNOSTICS = validate_scalar_states(probe_items, probe_vocab)
assert_candidate_and_metric_invariants(probe_items[probe_battery], probe_vocab)
width, pcount, gcount = matched_mlp_width(probe_vocab, probe_items[probe_battery]['local'].shape[-1])
print('Matched MLP width / parameters / GNN parameters:', width, pcount, gcount)
display(PREFLIGHT_DIAGNOSTICS)
probe_encoder = make_scalar_encoder(probe_vocab)
probe_rows = []
for mode in [False, True]:
    mlp, _ = train_scalar_mlp(probe_items, probe_train, probe_vocab, SEEDS[0], mode, epochs=1)
    metrics, _ = evaluate_scalar_or_gnn(mlp, probe_items[probe_battery], 'mlp', mode)
    reordered, _ = evaluate_scalar_or_gnn(mlp, probe_items[probe_battery], 'mlp', mode, candidate_seed=89)
    np.testing.assert_array_equal(list(metrics.values()), list(reordered.values()))
    probe_rows.append({'condition': 'StateAwareMLP' if mode else 'StaticFeatureMLP', **metrics})
    del mlp
    xx, yy, qq = training_xgb_matrix(probe_items, probe_train, probe_encoder, mode)
    tree = make_scalar_xgb(SEEDS[0], n_estimators=5)
    tree.fit(xx, yy, qid=qq, verbose=False)
    metrics, _ = evaluate_scalar_or_gnn(tree, probe_items[probe_battery], 'xgb', mode, probe_encoder)
    reordered, _ = evaluate_scalar_or_gnn(tree, probe_items[probe_battery], 'xgb', mode, probe_encoder, candidate_seed=89)
    np.testing.assert_array_equal(list(metrics.values()), list(reordered.values()))
    probe_rows.append({'condition': 'StateAwareXGB' if mode else 'StaticFeatureXGB', **metrics})
    del tree, xx, yy, qq
if RUN_DYNAMICREAL:
    saved_epochs = EPOCHS
    try:
        EPOCHS = 1
        graph_model, graph_item, _ = train_one_e6_condition(probe_train, probe_battery, SEEDS[0], 'dynamic_real')
        metrics, _ = evaluate_scalar_or_gnn(graph_model, graph_item, 'gnn')
        reordered, _ = evaluate_scalar_or_gnn(graph_model, graph_item, 'gnn', candidate_seed=89)
        np.testing.assert_array_equal(list(metrics.values()), list(reordered.values()))
        original_metrics, _ = evaluate_e6_model(graph_model, graph_item, probe_train)
        np.testing.assert_array_equal([metrics[m] for m in METRICS], [original_metrics[m] for m in METRICS])
        probe_rows.append({'condition': 'DynamicRealGNN', **metrics})
        del graph_model, graph_item
    finally:
        EPOCHS = saved_epochs
display(pd.DataFrame(probe_rows))
del probe_items
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
PREFLIGHT_PASSED = True
print('Preflight passed. These short-fit scores are not manuscript results.')

## Cell 10 — Full LOPO experiment with resume
This is the actual experiment. Leave it running until all fits finish.
Output folders are keyed by the data, feature implementation, hyperparameters,
and dependency versions. Incompatible settings cannot reuse old fits.
During a fit, progress appears at its start and completion; results are committed
after each fit. Re-running this cell skips validated completion files.

In [ ]:
def atomic_json(path, payload):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)

def atomic_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    compression = "gzip" if str(path).endswith(".gz") else None
    frame.to_csv(temp, index=False, compression=compression)
    os.replace(temp, path)

def protocol_payload():
    return {"experiment": "E9 state-aware non-graph baselines", "feature_version": FEATURE_VERSION,
            "implementation_sha256": IMPLEMENTATION_SHA256, "source_e6_sha256": SOURCE_E6_SHA256,
            "dataset_sha256": DATASET_HASHES, "seeds": SEEDS, "epochs": EPOCHS,
            "negative_samples": NEGATIVES_PER_STATE, "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY, "dropout": DROPOUT, "xgb_trees": XGB_TREES,
            "conditions": CONDITIONS, "folds": FOLD_IDS,
            "torch_version": torch.__version__, "xgboost_version": xgb.__version__,
            "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,
            "deterministic_torch": torch.are_deterministic_algorithms_enabled(),
            "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
            "evaluation_batching": "Canonical internal batching; shuffled candidate metrics; E6 tie tolerance 1e-8",
            "independent_unit": "held-out battery product",
            "state_rule": "Exact E6 post-current update; all original fixation classes retained",
            "mlp_capacity": "Width chosen by parameter-count gap only; shared by static/state models",
            "xgb_budget": "E3 250-tree settings; all nonsingleton training candidates, CPU hist",
            "notes": "No ID/order input, no test tuning; historic E6 is descriptive only"}

def prepare_run_directory():
    payload = protocol_payload()
    signature = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    run_dir = OUTPUT_ROOT / ("run_" + signature[:12])
    run_dir.mkdir(parents=True, exist_ok=True)
    for folder in ["fits", "models", "diagnostics"]:
        (run_dir/folder).mkdir(exist_ok=True)
    manifest_path = run_dir/"protocol.json"
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text())
        assert previous == payload, "Protocol mismatch: use a new output directory."
    else:
        atomic_json(manifest_path, payload)
    return run_dir, signature

def fit_paths(run_dir, battery_id, seed, condition):
    key = f"battery{int(battery_id):02d}_seed{int(seed)}_{condition}"
    return key, run_dir/"fits"/(key+".json"), run_dir/"fits"/(key+"_states.csv.gz")

def completed_fit(path, states_path, signature, battery_id, seed, condition):
    if not path.exists() or not states_path.exists():
        return False
    record = json.loads(path.read_text())
    if record["protocol_signature"] != signature:
        return False
    frame = pd.read_csv(states_path)
    assert len(frame) == len(chains[battery_id])-1 and frame[METRICS].notna().all().all()
    assert record["battery_id"] == battery_id and record["seed"] == seed and record["condition"] == condition
    return True

def run_e9():
    run_dir, signature = prepare_run_directory()
    total = len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS)
    print(f"Run directory: {run_dir}\nPlanned fits: {total}", flush=True)
    done = 0
    for held_out in FOLD_IDS:
        train = [b for b in sorted(chains) if b != held_out]
        assert held_out not in train and len(train) == 9
        vocab = build_fold_vocab(train)
        # Rebuilt after a restart; raw state summaries are small and deterministic.
        items = {b: build_scalar_state_pack(b, vocab) for b in sorted(chains)}
        n_local = items[held_out]["local"].shape[-1]
        width, count, target = matched_mlp_width(vocab, n_local)
        atomic_json(run_dir/"diagnostics"/f"features_fold{held_out:02d}.json",
                    {"vocab": vocab, "local_features": items[held_out]["local_names"],
                     "global_features": ["remaining_fixation_fraction", "active_fraction", "completed_fraction"],
                     "shared_features": ["any_shared_component", "shared_count_div2", "shared_jaccard"],
                     "mlp_width": width, "mlp_parameters": count, "gnn_parameters": target})
        encoder = make_scalar_encoder(vocab)
        for condition in CONDITIONS:
            state_aware = condition.startswith("StateAware")
            kind = "gnn" if condition == "DynamicRealGNN" else ("xgb" if condition.endswith("XGB") else "mlp")
            pending = []
            for seed in SEEDS:
                _, path, state_path = fit_paths(run_dir, held_out, seed, condition)
                if completed_fit(path, state_path, signature, held_out, seed, condition):
                    done += 1
                    print(f"[{done}/{total}] reuse {held_out} {seed} {condition}", flush=True)
                else:
                    pending.append(seed)
            matrix = None
            if kind == "xgb" and pending:
                matrix = training_xgb_matrix(items, train, encoder, state_aware)
                print(f"XGB fold {held_out}: {matrix[0].shape[0]:,} rows, {matrix[0].shape[1]} features", flush=True)
            for seed in pending:
                started = time.time()
                print(f"Starting fold {held_out}, seed {seed}, {condition}", flush=True)
                if kind == "mlp":
                    model, extra = train_scalar_mlp(items, train, vocab, seed, state_aware, epochs=EPOCHS)
                    test_item = items[held_out]
                elif kind == "xgb":
                    model = make_scalar_xgb(seed, n_estimators=XGB_TREES)
                    model.fit(matrix[0], matrix[1], qid=matrix[2], verbose=False)
                    extra = {"training_rows": int(matrix[0].shape[0]), "feature_count": int(matrix[0].shape[1])}
                    test_item = items[held_out]
                else:
                    model, test_item, _ = train_one_e6_condition(train, held_out, seed, "dynamic_real")
                    extra = {"parameter_count": count_trainable_parameters(model)}
                metrics, detail = evaluate_scalar_or_gnn(model, test_item, kind, state_aware, encoder)
                key, path, state_path = fit_paths(run_dir, held_out, seed, condition)
                if SAVE_MODELS:
                    if kind == "xgb":
                        model.save_model(run_dir/"models"/(key+".ubj"))
                    else:
                        torch.save({"state_dict": model.state_dict(), "vocab": vocab,
                                    "condition": condition, "extra": extra}, run_dir/"models"/(key+".pt"))
                detail["battery_id"] = held_out
                detail["battery"] = str(battery_names[held_out])
                detail["seed"] = seed
                detail["condition"] = condition
                record = {"protocol_signature": signature, "battery_id": held_out,
                          "battery": str(battery_names[held_out]), "seed": seed,
                          "condition": condition, "n_states": len(detail),
                          "runtime_seconds": time.time()-started,
                          "device": str(DEVICE), **metrics, **extra}
                # Detail first; JSON acts as the completion marker.
                atomic_csv(detail, state_path)
                atomic_json(path, record)
                done += 1
                print(f"[{done}/{total}] saved {condition}: Hits@1={metrics['hits1']:.4f}", flush=True)
                del model, test_item
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            del matrix
        del items
        gc.collect()
    print("All requested fits are saved.", flush=True)
    return run_dir

In [ ]:
if not globals().get('PREFLIGHT_PASSED', False):
    raise RuntimeError('Run Cell 9 successfully before launching the full experiment.')
if RUN_FULL_EXPERIMENT:
    RUN_DIR = run_e9()
    atomic_csv(PREFLIGHT_DIAGNOSTICS, RUN_DIR/'diagnostics'/'preflight_e6_state_match.csv')
else:
    RUN_DIR, _ = prepare_run_directory()
    print('Full run disabled. Set RUN_FULL_EXPERIMENT=True and rerun this cell when ready.')

## Cell 11 — Product-level summaries and prespecified comparisons
Average training seeds within product, then average the ten products equally.
Four planned primary Hits@1 contrasts share one Holm correction family:
DynamicReal minus each state-aware baseline, and each state-aware baseline minus
its static feature counterpart. Exact two-sided sign-flip values are reported only
for a complete ten-product, three-seed, 30-epoch/250-tree design. Bootstrap intervals
are descriptive. Seeds and states are not independent statistical units.
Early/middle/late and terminal-singleton analyses are descriptive supplements.

In [ ]:
def exact_sign_flip(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())
    means = [abs(np.mean(values*np.array(signs))) for signs in itertools.product([-1, 1], repeat=len(values))]
    return float(np.mean(np.asarray(means) >= observed-1e-12))

def holm(pvalues):
    values = np.asarray(pvalues, dtype=float)
    order = np.argsort(values)
    result = np.empty(len(values))
    last = 0.0
    for rank, idx in enumerate(order):
        last = max(last, min(1.0, (len(values)-rank)*values[idx]))
        result[idx] = last
    return result

def summarize_e9(run_dir):
    run_dir = Path(run_dir)
    records = [json.loads(path.read_text()) for path in sorted((run_dir/"fits").glob("*.json"))]
    if not records:
        raise RuntimeError("No finished fits. Run Cell 10 first.")
    all_results = pd.DataFrame([{k:v for k,v in rec.items() if k != "loss_curve"} for rec in records])
    assert not all_results.duplicated(["battery_id", "seed", "condition"]).any()
    expected_seeds = set(SEEDS)
    complete = []
    for (b, cond), group in all_results.groupby(["battery_id", "condition"]):
        if set(group.seed) == expected_seeds:
            complete.append({"battery_id": b, "battery": group.battery.iloc[0], "condition": cond,
                             **{m: float(group[m].mean()) for m in METRICS},
                             **{m+"_seed_sd": float(group[m].std(ddof=1)) for m in METRICS}})
    fold = pd.DataFrame(complete)
    atomic_csv(all_results, run_dir/"e9_fit_results.csv")
    atomic_csv(fold, run_dir/"e9_fold_mean.csv")
    aggregate = fold.groupby("condition", as_index=False).agg(
        hits1=("hits1", "mean"), hits3=("hits3", "mean"), mrr=("mrr", "mean"),
        n_products=("battery_id", "nunique"))
    atomic_csv(aggregate, run_dir/"e9_aggregate.csv")
    full_products = set(sorted(chains))
    pairs = [("DynamicRealGNN", "StateAwareMLP"), ("DynamicRealGNN", "StateAwareXGB"),
             ("StateAwareMLP", "StaticFeatureMLP"), ("StateAwareXGB", "StaticFeatureXGB")]
    contrasts = []
    for a, b in pairs:
        da = fold[fold.condition == a].set_index("battery_id")
        db = fold[fold.condition == b].set_index("battery_id")
        shared = sorted(set(da.index) & set(db.index))
        ready = (set(shared) == full_products and set(SEEDS) == {17,29,43}
                 and EPOCHS == 30 and XGB_TREES == 250)
        for metric in METRICS:
            difference = (da.loc[shared, metric]-db.loc[shared, metric]).to_numpy()
            if len(difference):
                rng = np.random.default_rng(BASE_SEED)
                boot = rng.choice(difference, (10000, len(difference)), replace=True).mean(axis=1)
                lower, upper = np.quantile(boot, [0.025, 0.975])
                mean, wins = float(difference.mean()), int((difference>0).sum())
            else:
                lower = upper = mean = np.nan
                wins = 0
            contrasts.append({"contrast": a+" - "+b, "metric": metric,
                               "n_products": len(shared), "mean_difference": mean,
                               "bootstrap95_low": lower, "bootstrap95_high": upper,
                               "wins": wins, "full_ten_product_design": ready,
                               "p_exact": exact_sign_flip(difference) if ready else np.nan})
    contrasts = pd.DataFrame(contrasts)
    primary = contrasts.metric == "hits1"
    # Four prespecified contrasts; skipped comparisons count as p=1.
    contrasts["p_holm"] = np.nan
    adjusted = holm(contrasts.loc[primary, "p_exact"].fillna(1.0))
    contrasts.loc[primary, "p_holm"] = np.where(contrasts.loc[primary, "p_exact"].notna(), adjusted, np.nan)
    atomic_csv(contrasts, run_dir/"e9_contrasts.csv")
    state_frames = [pd.read_csv(path) for path in sorted((run_dir/"fits").glob("*_states.csv.gz"))]
    details = pd.concat(state_frames, ignore_index=True)
    details["stage"] = pd.cut(details.progress, [-1, 1/3, 2/3, 1.001],
                              labels=["early", "middle", "late"], right=False)
    stages = details.groupby(["battery_id", "condition", "seed", "stage"], observed=True)[METRICS].mean().reset_index()
    stages = stages.groupby(["battery_id", "condition", "stage"], observed=True)[METRICS].mean().reset_index()
    atomic_csv(stages, run_dir/"e9_stage_product_metrics.csv")
    nonsingleton = details[details.candidate_size > 1]
    singleton = nonsingleton.groupby(["battery_id", "condition", "seed"])[METRICS].mean().reset_index()
    singleton = singleton.groupby(["battery_id", "condition"])[METRICS].mean().reset_index()
    atomic_csv(singleton, run_dir/"e9_without_singleton_fold.csv")
    expected = len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS)
    status = {"requested_fits": expected, "finished_fits": len(all_results),
              "full_canonical_design": set(SEEDS) == {17,29,43} and EPOCHS == 30 and XGB_TREES == 250
                                        and set(FOLD_IDS) == full_products,
              "all_requested_fits_complete": len(all_results) == expected,
              "historical_e6_included_in_inference": False}
    atomic_json(run_dir/"e9_completion.json", status)
    display(aggregate.sort_values("hits1", ascending=False))
    display(contrasts[primary])
    print("Completion:", status)
    if not status["full_canonical_design"]:
        print("DIAGNOSTIC RUN: these outputs are not the full manuscript experiment.")
    return aggregate, fold, contrasts, stages

In [ ]:
AGGREGATE, PRODUCT_RESULTS, CONTRASTS, STAGE_RESULTS = summarize_e9(RUN_DIR)
stage_summary = STAGE_RESULTS.groupby(['condition', 'stage'], observed=True)[METRICS].mean().reset_index()
display(stage_summary)
fig, ax = plt.subplots(figsize=(9, 4))
plot = AGGREGATE.sort_values('hits1')
ax.barh(plot.condition, plot.hits1)
ax.set_xlabel('Mean product-level Hits@1')
ax.set_xlim(0, max(0.5, float(plot.hits1.max())+0.05))
ax.set_title('E9: newly trained LOPO conditions')
fig.tight_layout()
fig.savefig(RUN_DIR/'e9_hits1.png', dpi=180)
plt.show()

## Cell 12 — Saved E6 reference (descriptive only)
The following 30 rows are copied from the uploaded repository's saved E6 outputs.
They are labelled **historical** and excluded from E9 inferential contrasts.
If DynamicReal rerunning is disabled, use this table only as context. These saved
rows are not substitutes for a newly matched comparator or independently verified
reproductions. No empirical baseline result is prefilled in this notebook.

In [ ]:
HISTORICAL_E6_CSV = 'battery_id,battery,seed,hits1,hits3,mrr\n1,Fiat 500e,17,0.3433785775201304,0.5127560534160063,0.4587206638686235\n1,Fiat 500e,29,0.3801581378291024,0.5458554295300881,0.4942952119095962\n1,Fiat 500e,43,0.3818478288570288,0.5319071229347229,0.4849325006121623\n2,Chevrolet Spark,17,0.4480411602337082,0.6584947381861549,0.5779472910308067\n2,Chevrolet Spark,29,0.4415541342856046,0.6360396483657956,0.5676442095836649\n2,Chevrolet Spark,43,0.4156060304931893,0.6390336603418435,0.5581164708378342\n3,Hyundai IONIQ 5,17,0.3698152163337211,0.4902098977058265,0.4630914461538844\n3,Hyundai IONIQ 5,29,0.3646662526031511,0.4709417629908006,0.4495295341702583\n3,Hyundai IONIQ 5,43,0.3788867327562737,0.5123078666626038,0.4795004471918767\n4,BMW 530e,17,0.4067439310513638,0.6138092878462359,0.5355501465766123\n4,BMW 530e,29,0.4259849076967991,0.6298655511158748,0.5543418946037263\n4,BMW 530e,43,0.3698543344483914,0.583421814385514,0.5051545696198556\n5,Chevrolet Volt,17,0.4996574134058196,0.6977674209403507,0.6189063747356865\n5,Chevrolet Volt,29,0.4757116302732896,0.6650866980487844,0.5867903579526502\n5,Chevrolet Volt,43,0.4061720433541846,0.5739378168095417,0.530618062209649\n6,Volvo EX30,17,0.3606031944512007,0.5468889484329674,0.4859140902083986\n6,Volvo EX30,29,0.3879841468321531,0.567523869067888,0.5079978585212523\n6,Volvo EX30,43,0.3606031944512008,0.5409365674805864,0.4877151759449997\n7,BMW 330e,17,0.3819808560992664,0.572865645220876,0.512682789942849\n7,BMW 330e,29,0.4105064971249074,0.6129297477849786,0.5407222812974455\n7,BMW 330e,43,0.3733270099454202,0.5674169272721582,0.5026252331638803\n8,MG ZS,17,0.3556997780904698,0.5151115666873115,0.4664024840448103\n8,MG ZS,29,0.3574836720761986,0.5129878833709677,0.4650902225789071\n8,MG ZS,43,0.3613667799089827,0.5411509683372097,0.4820656809211412\n9,Mitsubishi Outlander,17,0.3753759896343938,0.6078129505881632,0.5209972060345218\n9,Mitsubishi Outlander,29,0.3662184804768846,0.5840034267786393,0.5098405151764138\n9,Mitsubishi Outlander,43,0.360037161795566,0.6057525110277235,0.5126554439532272\n10,BMW i3,17,0.3790496341838278,0.5517806854972203,0.5012905591946727\n10,BMW i3,29,0.3722666884473938,0.525230297900321,0.4857513756648963\n10,BMW i3,43,0.3982356806954558,0.5721572081993244,0.5186336130311955\n'
historical = pd.read_csv(io.StringIO(HISTORICAL_E6_CSV))
historical['source'] = 'uploaded repository: saved E6; historical only'
atomic_csv(historical, RUN_DIR/'e9_historical_e6_reference.csv')
historical_product = historical.groupby('battery_id')[METRICS].mean()
print('Historical E6 macro average:', historical_product.mean().to_dict())
if 'DynamicRealGNN' in set(PRODUCT_RESULTS.condition):
    rerun = PRODUCT_RESULTS[PRODUCT_RESULTS.condition == 'DynamicRealGNN'].set_index('battery_id')
    comparison = rerun[METRICS].join(historical_product, lsuffix='_rerun', rsuffix='_historical')
    display(comparison)
    atomic_csv(comparison.reset_index(), RUN_DIR/'e9_rerun_vs_historical_descriptive.csv')

## Cell 13 — Export a result ZIP
The ZIP includes per-fit and per-state results, product means, contrasts,
stage analyses, singleton sensitivity, feature definitions, protocol hashes,
and saved models (if enabled). It does not include the raw KIT data.
Share `e9_state_aware_baselines_results.zip` for interpretation and manuscript updates.

If StateAwareMLP/XGB approach or exceed DynamicReal, narrow the graph-specific
contribution accordingly. If DynamicReal retains a consistent advantage, you
have stronger evidence beyond the local state summaries tested here. This notebook
does **not** resolve the separate activity-mask-versus-adjacency permutation issue.

In [ ]:
readme = """E9 state-aware non-graph baseline experiment

Read e9_completion.json before interpreting the results. e9_fit_results.csv holds
seed-level results; e9_fold_mean.csv averages seeds within product; e9_aggregate.csv
averages products equally. e9_contrasts.csv uses the four planned Hits@1 comparisons
with Holm correction. Hits@3/MRR, stage analyses, and singleton sensitivities are
supporting/descriptive. Product bootstrap intervals are descriptive. Exact sign-flip
enumeration relies on the product-level sign-exchangeability null assumption;
overlapping LOPO training sets and the small product collection limit population claims.

StaticFeature controls and StateAware controls have matched architecture/features;
only state summaries change. Shared-component indicators are present in both.
Non-graph does not mean no engineering-derived features: local fixation counts,
type histograms, and activity/completion flags are derived from the E6 state.
These learners receive no adjacency or message passing. DynamicReal is the original
E6 implementation rerun in this notebook. The old E6 CSV is historical context only.

No test-product hyperparameter selection, IDs, candidate positions, or future target
pointers enter model inputs. Prefix operations update state; candidates are inventory
minus executed operations and randomly ordered at evaluation. Fixed/no-operation
connections keep the original E6 handling, even if completed nodes remain active.

The raw dataset is external: DOI 10.35097/emz24pksshndq468. CSV hashes, source notebook
hash, dependency versions, feature names, vocabulary and parameter counts are saved.
"""
(RUN_DIR/'README_RESULTS.txt').write_text(readme)
zip_base = OUTPUT_ROOT/'e9_state_aware_baselines_results'
RESULT_ZIP = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=RUN_DIR))
print('Result ZIP:', RESULT_ZIP)
print('Size (MB):', RESULT_ZIP.stat().st_size/1e6)
if IN_COLAB:
    from google.colab import files
    files.download(str(RESULT_ZIP))